In [1]:
import sys
import numpy as np
sys.path.append('./src')
import importlib
import matplotlib.pyplot as plt
from dl_utils import *
from data_utils import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATH = r"data/df_sp_500_log_ret.csv"
# df = download_sp500_log_return_series()
# df.to_csv(DATA_PATH)
df = pd.read_csv(DATA_PATH, index_col='Date')

#number of lag to use for the prediction
LAGS = 15
#X, y = build_dataset_returns(df)
X, y = build_dataset_abs_returns(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=80718)
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_train = scaler_x.fit_transform(X_train)                                                                                                                                                                 
X_test = scaler_x.transform(X_test) 

y_train = scaler_y.fit_transform(y_train)                                                                                                                                                                 
y_test = scaler_y.transform(y_test) 

## Benchmark with Linear regression

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

lin_reg = LinearRegression().fit(X_train, y_train)
preds = lin_reg.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, preds):.4f}")

MAE: 0.6271


## First naive neural network, one layer of 10 with sgmoid activation

In [4]:
neural_network = NeuralNetwork(
                                layers=[Dense(neurons=10,
                                activation=Sigmoid()),
                                Dense(neurons=1,
                                activation=Linear())],
                                loss = MeanSquaredError()
)
trainer = Trainer(neural_network, SGD(lr=0.001))

trainer.fit(X_train, y_train, X_test, y_test,
       epochs = 100,
       eval_every = 10,
       seed=20190501)

nn_preds = neural_network.forward(X_test)                                                                                                                                                                 
print(f"NN  MAE: {np.mean(np.abs(nn_preds - y_test)):.4f}")                                                                                                                                               

Validation loss after 10 epochs is 1.473356
Validation loss after 20 epochs is 1.050871
Validation loss after 30 epochs is 0.929717
Validation loss after 40 epochs is 0.884463
Validation loss after 50 epochs is 0.864243
Validation loss after 60 epochs is 0.853859
Validation loss after 70 epochs is 0.847930
Validation loss after 80 epochs is 0.843794
Validation loss after 90 epochs is 0.840452
Validation loss after 100 epochs is 0.838107
NN  MAE: 0.6208


## Second neural network with different activation function

In [5]:
neural_network = NeuralNetwork(                                                                                                                                                                           
    layers=[                                                                                                                                                                                              
        Dense(neurons=32, activation=ReLU(), weight_init='he'),                                                                                                                                           
        Dense(neurons=16, activation=ReLU(), weight_init='he'),                                                                                                                                           
        Dense(neurons=1,  activation=Linear())  
    ],                                                                                                                                                                                                    
    loss=MeanSquaredError()                                                                                                                                                                             
) 
optim = SGD_MOMENTUM(lr=0.01, final_lr=0.001, decay_type='exponential')  
trainer = Trainer(neural_network, optim)
trainer.fit(X_train, y_train, X_test, y_test,                                                                                                                                                             
    epochs=300,                                                                                                                                                                                           
    eval_every=10,  
    seed = 201906501,                                                                                                                                                                                     
    patience=5)
nn_preds = neural_network.forward(X_test)                                                                                                                                                                 
print(f"NN  MAE: {np.mean(np.abs(nn_preds - y_test)):.4f}")    

Validation loss after 10 epochs is 0.737613
No improvement after epoch 20 (1/5), best loss: 0.737613
No improvement after epoch 30 (2/5), best loss: 0.737613
No improvement after epoch 40 (3/5), best loss: 0.737613
No improvement after epoch 50 (4/5), best loss: 0.737613
No improvement after epoch 60 (5/5), best loss: 0.737613
Early stopping triggered. Restoring best model.
NN  MAE: 0.5871


## Implementation from torch

In [6]:
import torch
import torch.nn as nn

# Same architecture
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

model = Net()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
loader  = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

for epoch in range(100):
    model.train()
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_preds = model(X_test_t).numpy()

mae_torch = np.mean(np.abs(test_preds - y_test))
print(f"PyTorch MAE: {mae_torch:.4f}")

PyTorch MAE: 0.5744
